# Generative adversarial networks — DCGAN

**DeepVision Masterclass — Modern Edition**

This notebook is designed as an executable teaching lab. Read the dataset contract and validation notes before training.

## 1. GAN objective
A generator maps latent noise to images; a discriminator distinguishes real from generated samples. GAN training is sensitive to architecture, optimizer and update balance.

In [ ]:
import torch, torchvision, matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
print(torch.__version__)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. MNIST pipeline
Scale pixels to [-1,1] to match the generator's Tanh output.

In [ ]:
ds=datasets.MNIST('data',train=True,download=True,transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize((.5,),(.5,))]))
loader=DataLoader(ds,batch_size=128,shuffle=True,drop_last=True)
latent_dim=100

## 3. Generator and discriminator
Transposed convolutions upsample the latent vector. The discriminator returns logits; use BCEWithLogitsLoss for numerical stability.

In [ ]:
class Generator(nn.Module):
 def __init__(self):
  super().__init__(); self.net=nn.Sequential(nn.Linear(100,128*7*7),nn.BatchNorm1d(128*7*7),nn.ReLU(),nn.Unflatten(1,(128,7,7)),nn.ConvTranspose2d(128,64,4,2,1),nn.BatchNorm2d(64),nn.ReLU(),nn.ConvTranspose2d(64,1,4,2,1),nn.Tanh())
 def forward(self,z): return self.net(z)
class Discriminator(nn.Module):
 def __init__(self):
  super().__init__(); self.net=nn.Sequential(nn.Conv2d(1,64,4,2,1),nn.LeakyReLU(.2),nn.Conv2d(64,128,4,2,1),nn.BatchNorm2d(128),nn.LeakyReLU(.2),nn.Flatten(),nn.Linear(128*7*7,1))
 def forward(self,x): return self.net(x)
G,D=Generator().to(device),Discriminator().to(device)
criterion=nn.BCEWithLogitsLoss(); optG=torch.optim.Adam(G.parameters(),lr=2e-4,betas=(.5,.999)); optD=torch.optim.Adam(D.parameters(),lr=2e-4,betas=(.5,.999))

## 4. Training loop
Use real labels=1 and fake labels=0. Track both losses and sample quality; a low discriminator loss alone does not mean a good GAN.

In [ ]:
epochs=5; fixed_noise=torch.randn(64,100,device=device)
for epoch in range(epochs):
 for real,_ in loader:
  real=real.to(device); b=real.size(0)
  optD.zero_grad(set_to_none=True)
  real_loss=criterion(D(real),torch.ones(b,1,device=device))
  fake=G(torch.randn(b,100,device=device)).detach()
  fake_loss=criterion(D(fake),torch.zeros(b,1,device=device))
  d_loss=real_loss+fake_loss; d_loss.backward(); optD.step()
  optG.zero_grad(set_to_none=True)
  gen=G(torch.randn(b,100,device=device))
  g_loss=criterion(D(gen),torch.ones(b,1,device=device)); g_loss.backward(); optG.step()
 print(f'Epoch {epoch+1}: D={d_loss.item():.3f} G={g_loss.item():.3f}')
with torch.no_grad(): samples=G(fixed_noise).cpu()

## 5. Inspect and save generated samples

In [ ]:
grid=torchvision.utils.make_grid(samples[:64],nrow=8,normalize=True)
plt.figure(figsize=(7,7)); plt.imshow(grid.permute(1,2,0)); plt.axis('off');
torch.save(G.state_dict(),'dcgan_generator.pt')

## 6. Keras implementation challenge
Rebuild the same generator/discriminator with `Conv2DTranspose`, `BatchNormalization`, `LeakyReLU` and a custom `tf.GradientTape` training step. Keep the discriminator output as logits and use `BinaryCrossentropy(from_logits=True)`.

## Exercises
Experiment with latent dimension, label smoothing, learning rates and spectral normalization. Discuss mode collapse, FID limitations on small datasets, and responsible use of generated imagery.